# Local Movie Q&A with Retrieval-Augmented Generation

**Overview.** A fully local retrieval-augmented generation (RAG) system that answers natural-language questions about movies. Answers must be grounded in retrieved documents and cite movie titles; questions the corpus cannot support get an exact refusal string. No hosted LLM, embedding, re-ranking or evaluation API is used.

**Dataset.** [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) (Kaggle; metadata and credits collected from TMDB). `movies_metadata.csv` (45,466 rows) and `credits.csv` are combined into a corpus of 41,328 movies with a July 2017 release cutoff.

**Approach.**
- One document per movie: title, year, genres, director, top-5 cast and overview
- Retrieval: `BAAI/bge-small-en-v1.5` embeddings in a FAISS `IndexFlatIP` (cosine), top-20 candidates
- Re-ranking: `cross-encoder/ms-marco-MiniLM-L-6-v2`, top-5 contexts
- Generation: `Qwen/Qwen2.5-3B-Instruct` in 4-bit NF4 with a strict grounded prompt, `[Title]` citations and a post-2017 year guard
- Evaluation: 20-query test set (standard, adversarial, sequel-confusion, unanswerable), Recall@k / MRR, three ablations, RAGAS with a local judge, and a failure diary

**Key results.**
- Cross-encoder re-ranking raises Recall@5 from 0.875 to 0.9375 on the 16 answerable queries
- The strict prompt reduces hallucinated answers on unanswerable queries from 4/4 to 0/4
- `bge-small` beats `bge-base` on re-ranked Recall@5 (0.9375 vs 0.875) with 134 s vs 364 s indexing time and half the embedding memory
- RAGAS on the strict prompt (6-query subset): faithfulness 0.75, context precision 0.625, answer relevancy 0.663

## 1. Setup & Imports

Designed to run top-to-bottom on a Colab T4 GPU runtime. Install dependencies with `pip install -r requirements.txt` and place `movies_metadata.csv` and `credits.csv` next to this notebook or in a `data/` subfolder.

In [ ]:
import ast
import gc
import json
import math
import os
import random
import re
import time
import warnings
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 455
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


Device: cuda


In [ ]:
def find_project_file(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path("/content") / filename,
        Path.cwd() / filename,
    ]
    for candidate in candidates:
        if candidate.exists():

            return candidate

    for root in [Path.cwd(), Path("/content")]:
        if root.exists():

            matches = list(root.rglob(filename))
            if matches:

                return matches[0]
    raise FileNotFoundError(f"Could not find {filename}. Put it next to this notebook.")


MOVIES_CSV = find_project_file("movies_metadata.csv")
CREDITS_CSV = find_project_file("credits.csv")


CUTOFF_DATE = pd.Timestamp("2017-07-31")

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
BASE_EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GENERATOR_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"



CPU_FALLBACK_GENERATOR_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
RAGAS_JUDGE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
RAGAS_JUDGE_MAX_NEW_TOKENS = 512


RETRIEVE_K = 20
RERANK_N = 5

EMBED_BATCH_SIZE = 128

BASE_EMBED_BATCH_SIZE = 64
CROSS_ENCODER_BATCH_SIZE = 32
GEN_MAX_NEW_TOKENS = 180

INDEX_ROOT = Path("rag_indexes")

INDEX_ROOT.mkdir(exist_ok=True)


REFUSAL_TEXT = "I don't have information about that in the provided context."


def safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name)


def free_memory():
    gc.collect()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


analysis_artifacts = {
    "seed": SEED,
    "retrieve_k": RETRIEVE_K,
    "rerank_n": RERANK_N,
    "embed_batch_size": EMBED_BATCH_SIZE,
    "base_embed_batch_size": BASE_EMBED_BATCH_SIZE,
    "cross_encoder_batch_size": CROSS_ENCODER_BATCH_SIZE,
    "generator_model": GENERATOR_MODEL_NAME,
    "ragas_judge_model": RAGAS_JUDGE_MODEL_NAME,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "base_embedding_model": BASE_EMBEDDING_MODEL_NAME,
    "cross_encoder_model": CROSS_ENCODER_MODEL_NAME,
    "vectordb": "FAISS IndexFlatIP over L2-normalized embeddings",
}

print("Movies CSV:", MOVIES_CSV)
print("Credits CSV:", CREDITS_CSV)


Movies CSV: movies_metadata.csv
Credits CSV: credits.csv


## 2. Data Pipeline

Preprocessing decisions:

- drop missing titles
- drop missing or empty overviews
- enforce the July 2017 corpus cutoff
- drop duplicate titles, keeping the row with the longer overview
- merge credits to add director and top cast names
- build one `document_text` field containing title, year, genres, director, cast, and overview

In [ ]:
def parse_jsonish_list(value):
    if isinstance(value, list):
        return value
    if not isinstance(value, str) or not value.strip():
        return []
    try:

        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):


        return []
    
    return parsed if isinstance(parsed, list) else []


def extract_names(value, limit=None):
    items = parse_jsonish_list(value)
    names = []

    for item in items:
        if isinstance(item, dict):
            name = str(item.get("name", "")).strip()
            if name:
                names.append(name)

    if limit is not None:

        names = names[:limit]

    return names



def extract_directors(value):
    items = parse_jsonish_list(value)
    directors = []
    for item in items:

        if isinstance(item, dict) and item.get("job") == "Director":
            name = str(item.get("name", "")).strip()
            if name:

                directors.append(name)
    return ", ".join(directors) if directors else "Unknown"


def list_to_text(values, empty="Unknown"):
    if isinstance(values, list) and values:

        return ", ".join(str(v) for v in values if str(v).strip())
    if isinstance(values, str) and values.strip():
        return values
    
    return empty


def build_document_text(row):
    year = "Unknown" if pd.isna(row["release_year"]) else str(int(row["release_year"]))
    genres = list_to_text(row["genres"], empty="Unknown")
    cast = list_to_text(row["cast_names"], empty="Unknown")
    director = row["director"] if isinstance(row["director"], str) and row["director"].strip() else "Unknown"

    return (
        
        f"Title: {row['title']}\n"
        f"Year: {year}\n"
        f"Genres: {genres}\n"
        f"Director: {director}\n"
        f"Cast: {cast}\n"
        f"Overview: {row['overview']}"
    )


def load_and_preprocess_movies(movies_csv: Path, credits_csv: Path) -> pd.DataFrame:
    movies = pd.read_csv(movies_csv, low_memory=False)
    credits = pd.read_csv(credits_csv)

    raw_movie_rows = len(movies)

    movies = movies.dropna(subset=["title"]).copy()
    movies["overview"] = movies["overview"].fillna("").astype(str)
    movies = movies[movies["overview"].str.strip().ne("")].copy()

    movies["movie_id"] = pd.to_numeric(movies["id"], errors="coerce")
    movies = movies.dropna(subset=["movie_id"]).copy()
    movies["movie_id"] = movies["movie_id"].astype("int64")

    movies["release_dt"] = pd.to_datetime(movies["release_date"], errors="coerce")
    movies = movies[(movies["release_dt"].isna()) | (movies["release_dt"] <= CUTOFF_DATE)].copy()
    movies["release_year"] = movies["release_dt"].dt.year.astype("Int64")

    movies["genres"] = movies["genres"].apply(lambda value: extract_names(value, limit=None))
    movies["overview_len_tokens"] = movies["overview"].str.split().str.len()

    movies = (
        movies.sort_values(["title", "overview_len_tokens"], ascending=[True, False])
        .drop_duplicates(subset=["title"], keep="first")
        .copy()
    )

    credits["movie_id"] = pd.to_numeric(credits["id"], errors="coerce")
    credits = credits.dropna(subset=["movie_id"]).copy()
    credits["movie_id"] = credits["movie_id"].astype("int64")
    credits["director"] = credits["crew"].apply(extract_directors)
    credits["cast_names"] = credits["cast"].apply(lambda value: extract_names(value, limit=5))
    credits = credits[["movie_id", "director", "cast_names"]].drop_duplicates(subset=["movie_id"])

    df = movies.merge(credits, on="movie_id", how="left")
    df["director"] = df["director"].fillna("Unknown")
    df["cast_names"] = df["cast_names"].apply(lambda value: value if isinstance(value, list) else [])
    df["document_text"] = df.apply(build_document_text, axis=1)
    df = df.reset_index(drop=True)
    df["row_id"] = df.index.astype(int)

    keep_cols = [
        "row_id",
        "movie_id",
        "title",
        "release_year",
        "genres",
        "director",
        "cast_names",
        "overview",
        "overview_len_tokens",
        "document_text",
    ]
    df = df[keep_cols].copy()

    analysis_artifacts["raw_movie_rows"] = raw_movie_rows
    analysis_artifacts["corpus_size"] = len(df)
    analysis_artifacts["overview_tokens_mean"] = float(df["overview_len_tokens"].mean())
    analysis_artifacts["overview_tokens_median"] = float(df["overview_len_tokens"].median())
    analysis_artifacts["overview_tokens_max"] = int(df["overview_len_tokens"].max())
    analysis_artifacts["document_text_format"] = (
        "Title, Year, Genres, Director, top-5 Cast, and Overview, one movie per document."
    )
    return df


movies_df = load_and_preprocess_movies(MOVIES_CSV, CREDITS_CSV)
movies_df.to_pickle("movies_preprocessed.pkl")

print(f"Final corpus size: {len(movies_df):,}")
print(
    "Overview length distribution in whitespace tokens: "
    f"mean={movies_df['overview_len_tokens'].mean():.2f}, "
    f"median={movies_df['overview_len_tokens'].median():.0f}, "
    f"max={movies_df['overview_len_tokens'].max():.0f}"
)
display(movies_df[["title", "release_year", "genres", "director", "overview_len_tokens"]].head())


Final corpus size: 41,328
Overview length distribution in whitespace tokens: mean=56.40, median=50, max=187


,title,release_year,genres,director,overview_len_tokens
0,!Women Art Revolution,2010,[Documentary],Lynn Hershman Leeson,38
1,#1 Cheerleader Camp,2010,"[Comedy, Drama]",Mark Quod,14
2,#Horror,2015,"[Drama, Mystery, Horror, Thriller]",Tara Subkoff,37
3,#chicagoGirl,2013,[Documentary],Joe Piscatella,84
4,"$1,000 on the Black",1966,[Western],Alberto Cardone,77


In [ ]:
print("Three example document_text strings from the corpus:\n")
for idx in [0, len(movies_df) // 2, len(movies_df) - 1]:
    print("=" * 90)
    print(movies_df.loc[idx, "document_text"][:1400])
    print()


Three example document_text strings from the corpus:

Title: !Women Art Revolution
Year: 2010
Genres: Documentary
Director: Lynn Hershman Leeson
Cast: Lynn Hershman Leeson
Overview: Through intimate interviews, provocative art, and rare, historical film and video footage, this feature documentary reveals how art addressing political consequences of discrimination and violence, the Feminist Art Revolution radically transformed the art and culture of our times.

Title: Mothra vs. Godzilla
Year: 1964
Genres: Fantasy, Science Fiction, Action, Adventure
Director: Ishirô Honda
Cast: Akira Takarada, Yuriko Hoshi, Hiroshi Koizumi, Yû Fujiki, Emi Ito
Overview: The 4th film in the Godzilla franchise restores the serious tone of the original Godzilla film. A greedy developer has placed huge machines to suck dry a part of the ocean near Tokyo so he can build luxury condos. After a storm, a giant egg washes up on a nearby beach.  The developer's plans go awry when he disrupts Godzilla's rest and th

## 3. Embedding & Indexing

I use FAISS because this corpus is small enough for exact search and FAISS makes latency/throughput easy to inspect. Vectors are L2-normalized and stored in `IndexFlatIP`, so inner product is cosine similarity.

In [ ]:
def json_default(value):

    if isinstance(value, (np.integer,)):
        return int(value)
    
    if isinstance(value, (np.floating,)):
        return float(value)
    
    if pd.isna(value) if not isinstance(value, (list, dict, str)) else False:
        return None
    
    return str(value)


def write_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8") as f:

        for row in rows:

            f.write(json.dumps(row, ensure_ascii=False, default=json_default) + "\n")


def read_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def records_from_df(df: pd.DataFrame):
    record_cols = [
        "row_id",
        "movie_id",
        "title",
        "release_year",
        "genres",
        "director",
        "cast_names",
        "overview",
        "document_text",
    ]
    clean_df = df[record_cols].copy()
    clean_df["release_year"] = clean_df["release_year"].astype("object")
    clean_df = clean_df.where(pd.notnull(clean_df), None)
    return clean_df.to_dict("records")


def build_or_load_faiss_index(
    model_name: str,
    df: pd.DataFrame,

    index_label: str,
    batch_size: int = EMBED_BATCH_SIZE,
    force_rebuild: bool = False,
):
    index_dir = INDEX_ROOT / safe_name(index_label)
    index_dir.mkdir(parents=True, exist_ok=True)

    index_path = index_dir / "index.faiss"

    records_path = index_dir / "records.jsonl"
    stats_path = index_dir / "stats.json"

    print(f"Loading bi-encoder: {model_name}")
    model = SentenceTransformer(model_name, device=DEVICE)

    if index_path.exists() and records_path.exists() and stats_path.exists() and not force_rebuild:
        
        print(f"Loading cached FAISS index from {index_path}")
        
        index = faiss.read_index(str(index_path))
        
        records = read_jsonl(records_path)
        
        stats = json.loads(stats_path.read_text())
        
        return index, records, stats, model

    texts = df["document_text"].tolist()
    start = time.perf_counter()
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype("float32")
    encode_seconds = time.perf_counter() - start

    dim = embeddings.shape[1]
    
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss.write_index(index, str(index_path))
    
    records = records_from_df(df)
    write_jsonl(records_path, records)

    stats = {
        "model_name": model_name,
        "index_label": index_label,
        "num_vectors": int(index.ntotal),
        "dim": int(dim),
        "encoding_time_sec": float(encode_seconds),
        "encoding_throughput_docs_per_sec": float(len(texts) / max(encode_seconds, 1e-9)),
        "embedding_memory_gb": float(embeddings.nbytes / (1024 ** 3)),
        "faiss_index_path": str(index_path),
        "records_path": str(records_path),
        "normalized_embeddings": True,
        "faiss_index_type": "IndexFlatIP",
    }
    stats_path.write_text(json.dumps(stats, indent=2))

    print(
        f"Encoded {len(texts):,} docs in {encode_seconds:.2f}s "
        f"({stats['encoding_throughput_docs_per_sec']:.1f} docs/sec)."
    )
    print(f"Embedding matrix memory: {stats['embedding_memory_gb']:.3f} GB")

    del embeddings
    free_memory()
    return index, records, stats, model


primary_index, movie_records, primary_index_stats, bi_encoder = build_or_load_faiss_index(
    EMBEDDING_MODEL_NAME,
    movies_df,
    index_label=f"primary_{EMBEDDING_MODEL_NAME}",
    batch_size=EMBED_BATCH_SIZE,
    force_rebuild=False,
)

analysis_artifacts["primary_index_stats"] = primary_index_stats
display(pd.DataFrame([primary_index_stats]))


Loading bi-encoder: BAAI/bge-small-en-v1.5


Encoded 41,328 docs in 134.37s (307.6 docs/sec).
Embedding matrix memory: 0.059 GB


,model_name,index_label,num_vectors,dim,encoding_time_sec,encoding_throughput_docs_per_sec,embedding_memory_gb,faiss_index_path,records_path,normalized_embeddings,faiss_index_type
0,BAAI/bge-small-en-v1.5,primary_BAAI/bge-small-en-v1.5,41328,384,134.368711,307.571604,0.05912,rag_indexes/primary_BAAI_bge-small-en-v1.5/ind...,rag_indexes/primary_BAAI_bge-small-en-v1.5/rec...,True,IndexFlatIP


In [ ]:
def retrieve_with_components(query: str, index, records, model, k: int = RETRIEVE_K):
    if not isinstance(query, str) or not query.strip():
        return []
    
    k = max(1, min(int(k), index.ntotal))

    q_emb = model.encode(

        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    scores, idxs = index.search(q_emb, k)
    results = []
    
    for rank, (idx, score) in enumerate(zip(idxs[0], scores[0]), start=1):
        if idx < 0:
            continue
        record = dict(records[int(idx)])
        record["rank"] = rank
        record["score"] = float(score)
        results.append(record)
    return results


def retrieve(query: str, k: int = RETRIEVE_K):
    return retrieve_with_components(query, primary_index, movie_records, bi_encoder, k=k)


sanity_query = "Who directed Inception?"
sanity_results = retrieve(sanity_query, k=5)
print(f"Sanity query: {sanity_query}")
display(pd.DataFrame([{ "rank": r["rank"], "title": r["title"], "year": r["release_year"], "score": r["score"] } for r in sanity_results]))


Sanity query: Who directed Inception?


,rank,title,year,score
0,1,Inception,2010,0.755771
1,2,Dream Work,2001,0.662704
2,3,Room 237,2012,0.651528
3,4,The American Dreamer,1971,0.643448
4,5,Dreams on Spec,2007,0.640640


## 4. Retrieval + Re-ranking

The bi-encoder retrieves a candidate pool. The cross-encoder then scores each `(query, document)` pair jointly and re-ranks the pool to the top 5 contexts used by the generator.

In [ ]:
cross_encoder = None


def ensure_cross_encoder():
    global cross_encoder
    if cross_encoder is None:
        print(f"Loading cross-encoder: {CROSS_ENCODER_MODEL_NAME}")
        cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL_NAME, device=DEVICE)
    return cross_encoder


def rerank(query: str, candidates, n: int = RERANK_N, cross_model=None):
    if not candidates:

        return []
    model = cross_model if cross_model is not None else ensure_cross_encoder()
    pairs = [(query, candidate["document_text"]) for candidate in candidates]
    scores = model.predict(pairs, batch_size=CROSS_ENCODER_BATCH_SIZE)
    reranked = []
    for candidate, score in zip(candidates, scores):
        updated = dict(candidate)
        updated["rerank_score"] = float(score)
        reranked.append(updated)
    reranked.sort(key=lambda row: row["rerank_score"], reverse=True)
    for rank, row in enumerate(reranked[:n], start=1):
        row["rerank_rank"] = rank
    return reranked[:n]


def search(query: str, k: int = RETRIEVE_K, n: int = RERANK_N):
    candidates = retrieve(query, k=k)
    return rerank(query, candidates, n=n)


comparison_queries = [
    "Who directed Toy Story?",
    "Which film is about a thief who enters dreams to steal secrets?",
    "What movie is about a shark terrorizing Amity Island?",
]

for q in comparison_queries:
    print("=" * 100)
    print("Query:", q)

    bi_top5 = retrieve(q, k=5)
    
    reranked_top5 = rerank(q, retrieve(q, k=20), n=5)
    bi_df = pd.DataFrame(
        [{"rank": r["rank"], "title": r["title"], "year": r["release_year"], "bi_score": r["score"]} for r in bi_top5]
    )
    rerank_df = pd.DataFrame(
        [
            {
                "rank": r["rerank_rank"],
                "title": r["title"],
                "year": r["release_year"],
                "bi_score": r["score"],
                "rerank_score": r["rerank_score"],
            }
            for r in reranked_top5
        ]
    )
    print("Bi-encoder top 5")
    display(bi_df)
    print("After cross-encoder re-ranking")
    display(rerank_df)


Query: Who directed Toy Story?
Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Bi-encoder top 5


,rank,title,year,bi_score
0,1,Toy Story 3,2010,0.724171
1,2,Toy Story 2,1999,0.707505
2,3,Toy Story,1995,0.695470
3,4,Toy Story of Terror!,2013,0.673922
4,5,The Pixar Story,2007,0.668802


After cross-encoder re-ranking


,rank,title,year,bi_score,rerank_score
0,1,Toy Story,1995,0.695470,7.910244
1,2,Toy Story 2,1999,0.707505,7.859396
2,3,Toy Story 3,2010,0.724171,7.475904
3,4,Toy Story That Time Forgot,2014,0.658232,6.929031
4,5,Toy Story of Terror!,2013,0.673922,6.733525


...

,rank,title,year,bi_score,rerank_score
0,1,To Steal from a Thief,2016,0.644471,1.078408
1,2,Seven Thieves,1960,0.640867,-1.656991
2,3,Thieves,2007,0.698452,-1.867778
3,4,Once a Thief,1996,0.649140,-2.050047
4,5,Stolen,2012,0.643455,-2.093719


Query: What movie is about a shark terrorizing Amity Island?
Bi-encoder top 5


,rank,title,year,bi_score
0,1,Jaws,1975,0.791462
1,2,Shark Attack 3: Megalodon,2002,0.732470
2,3,Shark Attack,1999,0.709284
3,4,Jaws 2,1978,0.705187
4,5,Shark Attack 2,2001,0.700747


After cross-encoder re-ranking


,rank,title,year,bi_score,rerank_score
0,1,Jaws,1975,0.791462,6.692847
1,2,Jaws 2,1978,0.705187,4.284674
2,3,Jaws: The Revenge,1987,0.696414,4.149043
3,4,Sharktopus,2010,0.667522,0.542806
4,5,Shark in Venice,2008,0.667140,0.150475


## 5. Generation

Generation uses a local open-weight instruction model. The strict prompt forbids outside knowledge, requires refusal when context is insufficient, and asks for title citations.

In [ ]:
PERMISSIVE_SYSTEM_PROMPT = """Use the provided movie context to answer the user's question. Cite movie titles when they are useful."""

STRICT_SYSTEM_PROMPT = """You are a grounded movie question-answering assistant.

Rules:
1. Answer ONLY using the provided movie context.
2. If the context does not contain enough evidence, answer exactly: "I don't have information about that in the provided context."
3. Do not use outside knowledge, even if you know the movie.
4. Cite every movie title you used in square brackets, for example [Toy Story].
5. If the user asks about a movie released after July 2017, refuse unless that exact movie appears in the context.

Refusal example:
Question: Who directed Oppenheimer (2023)?
Answer: I don't have information about that in the provided context.
"""


def concise_doc(record, max_overview_chars=850):
    year = record.get("release_year") or "Unknown"
    genres = list_to_text(record.get("genres", []), empty="Unknown")
    cast = list_to_text(record.get("cast_names", []), empty="Unknown")
    overview = str(record.get("overview", ""))[:max_overview_chars]
    return (
        f"Title: {record.get('title', 'Unknown')}\n"
        f"Year: {year}\n"
        f"Genres: {genres}\n"
        f"Director: {record.get('director', 'Unknown')}\n"
        f"Cast: {cast}\n"
        f"Overview: {overview}"
    )


def format_context(records, max_docs=RERANK_N):
    chunks = []
    for i, record in enumerate(records[:max_docs], start=1):
        chunks.append(f"[Document {i}]\n{concise_doc(record)}")
    return "\n\n".join(chunks)


def format_context_overview_only(records, max_docs=RERANK_N):
    chunks = []
    for i, record in enumerate(records[:max_docs], start=1):
        chunks.append(
            f"[Document {i}]\n"
            f"Title: {record.get('title', 'Unknown')}\n"
            f"Overview: {str(record.get('overview', ''))[:850]}"
        )
    return "\n\n".join(chunks)


def mentions_post_cutoff_year(query: str) -> bool:
    years = [int(match) for match in re.findall(r"\b(?:19|20)\d{2}\b", str(query))]
    return any(year > 2017 for year in years)


generator_tokenizer = None
generator_model = None
GENERATOR_MODEL_USED = None


def ensure_generator():
    global generator_tokenizer, generator_model, GENERATOR_MODEL_USED
    if generator_model is not None and generator_tokenizer is not None:
        return generator_tokenizer, generator_model

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    model_name = GENERATOR_MODEL_NAME if torch.cuda.is_available() else CPU_FALLBACK_GENERATOR_MODEL_NAME
    GENERATOR_MODEL_USED = model_name
    print(f"Loading local generator: {model_name}")
    generator_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if generator_tokenizer.pad_token is None:
        generator_tokenizer.pad_token = generator_tokenizer.eos_token

    if torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        generator_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        generator_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            trust_remote_code=True,
        )
        generator_model.to("cpu")
    generator_model.eval()
    return generator_tokenizer, generator_model


def unload_generator():
    global generator_tokenizer, generator_model
    generator_tokenizer = None
    generator_model = None
    free_memory()


def postprocess_answer(answer: str, contexts):
    answer = (answer or "").strip()
    if not answer:
        return REFUSAL_TEXT
    if REFUSAL_TEXT.lower() in answer.lower():
        return REFUSAL_TEXT
    cited = any(f"[{record['title']}]" in answer for record in contexts)
    if contexts and not cited:
        answer = f"{answer} [{contexts[0]['title']}]"
    return answer


def generate_answer(
    query: str,
    contexts,
    system_prompt: str = STRICT_SYSTEM_PROMPT,
    max_new_tokens: int = GEN_MAX_NEW_TOKENS,
    context_text: str | None = None,
):
    if not contexts:
        return REFUSAL_TEXT

    tokenizer, model = ensure_generator()
    context_text = context_text if context_text is not None else format_context(contexts)
    user_prompt = (
        "Movie context:\n"
        f"{context_text}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = system_prompt + "\n\n" + user_prompt

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3900)
    target_device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {key: value.to(target_device) for key, value in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1] :]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return postprocess_answer(answer, contexts)


def run_rag_pipeline(
    query: str,
    k: int = RETRIEVE_K,
    n: int = RERANK_N,
    system_prompt: str = STRICT_SYSTEM_PROMPT,
    use_year_guard: bool = True,
):
    if not isinstance(query, str) or not query.strip():
        return {
            "answer": REFUSAL_TEXT,
            "retrieved_titles": [],
            "contexts": [],
            "contexts_text": [],
        }
    candidates = retrieve(query, k=k)
    top_contexts = rerank(query, candidates, n=n)
    retrieved_titles = [record["title"] for record in top_contexts]

    if use_year_guard and mentions_post_cutoff_year(query):
        answer = REFUSAL_TEXT
    else:
        answer = generate_answer(query, top_contexts, system_prompt=system_prompt)

    return {
        "answer": answer,
        "retrieved_titles": retrieved_titles,
        "contexts": top_contexts,
        "contexts_text": [concise_doc(record) for record in top_contexts],
    }


generation_examples = [
    "Who directed Inception?",
    "Who directed Toy Story?",
    "What is the plot of The Matrix?",
    "Which movie is about a clownfish named Nemo?",
    "Who directed Oppenheimer (2023)?",
]

example_outputs = []
for query in generation_examples:
    result = run_rag_pipeline(query)
    example_outputs.append({"query": query, "answer": result["answer"], "retrieved_titles": result["retrieved_titles"]})
display(pd.DataFrame(example_outputs))
analysis_artifacts["generator_model_used"] = GENERATOR_MODEL_USED


Loading local generator: Qwen/Qwen2.5-3B-Instruct


,query,answer,retrieved_titles
0,Who directed Inception?,The director of Inception was Christopher Nola...,"[Inception, Transcendent Man, Directed by Sidn..."
1,Who directed Toy Story?,The director of Toy Story is John Lasseter. [T...,"[Toy Story, Toy Story 2, Toy Story 3, Toy Stor..."
2,What is the plot of The Matrix?,The plot of The Matrix involves a computer hac...,"[The Matrix, The Matrix Revolutions, The Matri..."
3,Which movie is about a clownfish named Nemo?,The movie about a clownfish named Nemo is Find...,"[Finding Nemo, Captain Nemo and the Underwater..."
4,Who directed Oppenheimer (2023)?,I don't have information about that in the pro...,"[The Day After Trinity, Into the Inferno, Tar,..."


## 6. `run_query` Entry Point

Single public entry point for the pipeline. It always returns exactly two keys, `answer` and `retrieved_titles`, and catches internal errors so callers never see an exception.

In [ ]:
def run_query(query: str) -> dict:
    """
    Answer a movie question with the full retrieve -> re-rank -> generate pipeline.

    Args:
        query: A natural language question (string).

    Returns:
        A dict with EXACTLY these two keys:
            "answer": str
            "retrieved_titles": list[str]
    """
    try:
        result = run_rag_pipeline(str(query) if query is not None else "")
        answer = result.get("answer") or REFUSAL_TEXT
        retrieved_titles = result.get("retrieved_titles") or []
        retrieved_titles = [str(title) for title in retrieved_titles]
        return {"answer": str(answer), "retrieved_titles": retrieved_titles}
    except Exception as exc:
        return {"answer": f"{REFUSAL_TEXT} Internal error: {type(exc).__name__}", "retrieved_titles": []}


test_inputs = [
    "Who directed Inception?",
    "asdfgh nonsense query",
    "",
    "Bir Zamanlar Anadolu'da hakkinda ne biliyorsun?",
]

for q in test_inputs:
    r = run_query(q)
    assert isinstance(r, dict), f"must return dict, got {type(r)}"
    assert "answer" in r and "retrieved_titles" in r, f"missing keys: {list(r.keys())}"
    assert isinstance(r["answer"], str), "answer must be str"
    assert isinstance(r["retrieved_titles"], list), "retrieved_titles must be list"
    assert all(isinstance(t, str) for t in r["retrieved_titles"]), "all titles must be strings"
    print(f"OK: {q[:40]!r} -> {len(r['retrieved_titles'])} titles, ans={r['answer'][:80]!r}")

OK: 'Who directed Inception?' -> 5 titles, ans='The director of Inception was Christopher Nolan. [Inception]'
OK: 'asdfgh nonsense query' -> 5 titles, ans="I don't have information about that in the provided context."
OK: '' -> 0 titles, ans="I don't have information about that in the provided context."
OK: "Bir Zamanlar Anadolu'da hakkinda ne bili" -> 5 titles, ans="I don't have information about that in the provided context."


## 7. Ablations

The 20-query test set (standard, adversarial, sequel-confusion and unanswerable questions) is defined here because the ablations use it directly. It is shown again with reference answers in Section 8.

In [ ]:
TEST_SET = [
    {
        "query": "Who directed Inception?",
        "gold_titles": ["Inception"],
        "category": "standard",
        "reference_answer": "Christopher Nolan directed Inception.",
    },
    {
        "query": "What is the plot of The Matrix?",
        "gold_titles": ["The Matrix"],
        "category": "standard",
        "reference_answer": "The Matrix is about a computer hacker who learns reality is a simulated world and joins rebels fighting its controllers.",
    },
    {
        "query": "Who directed Pulp Fiction?",
        "gold_titles": ["Pulp Fiction"],
        "category": "standard",
        "reference_answer": "Quentin Tarantino directed Pulp Fiction.",
    },
    {
        "query": "Which movie features a clownfish named Nemo?",
        "gold_titles": ["Finding Nemo"],
        "category": "standard",
        "reference_answer": "Finding Nemo features the clownfish Nemo.",
    },
    {
        "query": "Which movie follows a young lion prince reclaiming his kingdom after his father dies?",
        "gold_titles": ["The Lion King"],
        "category": "standard",
        "reference_answer": "The Lion King follows a young lion prince reclaiming his kingdom.",
    },
    {
        "query": "Which film is about dreams, corporate secrets, and a thief who enters people's minds?",
        "gold_titles": ["Inception"],
        "category": "adversarial",
        "reference_answer": "Inception is about dreams, corporate secrets, and a thief who enters minds.",
    },
    {
        "query": "What movie is about romance and disaster aboard the doomed ship Titanic?",
        "gold_titles": ["Titanic"],
        "category": "adversarial",
        "reference_answer": "Titanic is the romance and disaster movie aboard the doomed ship.",
    },
    {
        "query": "Which film has a ring that must be destroyed in Mount Doom?",
        "gold_titles": ["The Lord of the Rings: The Fellowship of the Ring"],
        "category": "adversarial",
        "reference_answer": "The Lord of the Rings: The Fellowship of the Ring involves the quest to destroy the ring.",
    },
    {
        "query": "What movie is about a shark terrorizing Amity Island?",
        "gold_titles": ["Jaws"],
        "category": "adversarial",
        "reference_answer": "Jaws is about a shark terrorizing Amity Island.",
    },
    {
        "query": "Which film features a chocolate factory run by Willy Wonka?",
        "gold_titles": ["Willy Wonka & the Chocolate Factory"],
        "category": "adversarial",
        "reference_answer": "Willy Wonka & the Chocolate Factory features Willy Wonka's chocolate factory.",
    },
    {
        "query": "Who directed Alien?",
        "gold_titles": ["Alien"],
        "category": "sequel_confusion",
        "reference_answer": "Ridley Scott directed Alien.",
    },
    {
        "query": "Who directed Toy Story?",
        "gold_titles": ["Toy Story"],
        "category": "sequel_confusion",
        "reference_answer": "John Lasseter directed Toy Story.",
    },
    {
        "query": "What is the plot of The Godfather?",
        "gold_titles": ["The Godfather"],
        "category": "sequel_confusion",
        "reference_answer": "The Godfather follows the aging patriarch of an organized crime dynasty transferring control to his reluctant son.",
    },
    {
        "query": "Who directed Terminator 2: Judgment Day?",
        "gold_titles": ["Terminator 2: Judgment Day"],
        "category": "standard",
        "reference_answer": "James Cameron directed Terminator 2: Judgment Day.",
    },
    {
        "query": "Which animated movie is about toys that come alive when humans are not around?",
        "gold_titles": ["Toy Story"],
        "category": "adversarial",
        "reference_answer": "Toy Story is about toys that come alive when humans are not around.",
    },
    {
        "query": "Who directed Oppenheimer (2023)?",
        "gold_titles": [],
        "category": "unanswerable",
        "reference_answer": "This is not answerable from the pre-July 2017 movie corpus.",
    },
    {
        "query": "What is the plot of Dune: Part Two?",
        "gold_titles": [],
        "category": "unanswerable",
        "reference_answer": "This is not answerable from the pre-July 2017 movie corpus.",
    },
    {
        "query": "Who stars in Everything Everywhere All at Once?",
        "gold_titles": [],
        "category": "unanswerable",
        "reference_answer": "This is not answerable from the pre-July 2017 movie corpus.",
    },
    {
        "query": "What is the story of the fictional film Moon Castle 2099?",
        "gold_titles": [],
        "category": "unanswerable",
        "reference_answer": "This is not answerable from the pre-July 2017 movie corpus.",
    },
    {
        "query": "Which movie follows archaeologist Indiana Jones searching for the Ark of the Covenant?",
        "gold_titles": ["Raiders of the Lost Ark"],
        "category": "adversarial",
        "reference_answer": "Raiders of the Lost Ark follows Indiana Jones searching for the Ark of the Covenant.",
    },
]

test_set_df = pd.DataFrame(TEST_SET)

display(test_set_df[["category", "query", "gold_titles"]])


,category,query,gold_titles
0,standard,Who directed Inception?,[Inception]
1,standard,What is the plot of The Matrix?,[The Matrix]
2,standard,Who directed Pulp Fiction?,[Pulp Fiction]
3,standard,Which movie features a clownfish named Nemo?,[Finding Nemo]
4,standard,Which movie follows a young lion prince reclai...,[The Lion King]
5,adversarial,"Which film is about dreams, corporate secrets,...",[Inception]
6,adversarial,What movie is about romance and disaster aboar...,[Titanic]
7,adversarial,Which film has a ring that must be destroyed i...,[The Lord of the Rings: The Fellowship of the ...
8,adversarial,What movie is about a shark terrorizing Amity ...,[Jaws]
9,adversarial,Which film features a chocolate factory run by...,[Willy Wonka & the Chocolate Factory]


In [ ]:
def answerable_items(test_set=TEST_SET):
    return [item for item in test_set if item["gold_titles"]]


def reciprocal_rank(ranked_titles, gold_titles):
    gold = set(gold_titles)
    
    for rank, title in enumerate(ranked_titles, start=1):
        if title in gold:
            return 1.0 / rank
    return 0.0


def recall_at_k(ranked_titles, gold_titles, k):
    if not gold_titles:
        return np.nan
    return float(any(title in set(gold_titles) for title in ranked_titles[:k]))


def summarize_retrieval_metrics(title_lists_by_query, test_set=TEST_SET, ks=(5, 10), mrr_k=None):
    rows = []

    for item in answerable_items(test_set):

        ranked_titles = title_lists_by_query[item["query"]]
        row = {"query": item["query"], "category": item["category"]}
        for k in ks:

            row[f"Recall@{k}"] = recall_at_k(ranked_titles, item["gold_titles"], k)
        cutoff_titles = ranked_titles if mrr_k is None else ranked_titles[:mrr_k]
        row["MRR"] = reciprocal_rank(cutoff_titles, item["gold_titles"])
        rows.append(row)


    per_query = pd.DataFrame(rows)
    summary = per_query.drop(columns=["query", "category"]).mean(numeric_only=True).to_dict()
    return summary, per_query


def median_latency_ms(fn, queries):

    latencies = []
    for query in queries:
        start = time.perf_counter()

        fn(query)
        latencies.append((time.perf_counter() - start) * 1000)
    return float(np.median(latencies))


def titles_from_bi(query, k=10):

    return [record["title"] for record in retrieve(query, k=k)]


def titles_from_reranker(query, k=RETRIEVE_K, n=RERANK_N):

    return [record["title"] for record in rerank(query, retrieve(query, k=k), n=n)]


### 7A. Ablation A: Bi-encoder Size

Small and base BGE models are compared with the same FAISS configuration and the same cross-encoder re-ranker.


In [ ]:
def evaluate_biencoder_model(model_name, batch_size):
    if model_name == EMBEDDING_MODEL_NAME:
        index = primary_index
        records = movie_records
        model = bi_encoder
        stats = primary_index_stats
        delete_model_after = False
    else:
        index, records, stats, model = build_or_load_faiss_index(
            model_name,
            movies_df,
            index_label=f"ablation_{model_name}",
            batch_size=batch_size,
            force_rebuild=False,
        )
        delete_model_after = True

    ensure_cross_encoder()

    title_lists = {}
    for item in TEST_SET:
        candidates = retrieve_with_components(item["query"], index, records, model, k=RETRIEVE_K)
        ranked = rerank(item["query"], candidates, n=RERANK_N)
        title_lists[item["query"]] = [record["title"] for record in ranked]

    summary, per_query = summarize_retrieval_metrics(title_lists, ks=(5,), mrr_k=5)

    def one_query(query):
        candidates = retrieve_with_components(query, index, records, model, k=RETRIEVE_K)
        return rerank(query, candidates, n=RERANK_N)

    latency = median_latency_ms(one_query, [item["query"] for item in TEST_SET])
    row = {
        "model": model_name,
        "indexing_time_sec": stats.get("encoding_time_sec", np.nan),
        "embedding_memory_gb": stats.get("embedding_memory_gb", np.nan),
        "R@5_reranked": summary.get("Recall@5", np.nan),
        "MRR_reranked": summary.get("MRR", np.nan),
        "median_latency_ms": latency,
    }

    if delete_model_after:
        del model
        free_memory()

    return row, per_query


ablation_a_rows = []
ablation_a_per_query = {}
for model_name, batch_size in [
    (EMBEDDING_MODEL_NAME, EMBED_BATCH_SIZE),
    (BASE_EMBEDDING_MODEL_NAME, BASE_EMBED_BATCH_SIZE),
]:
    row, per_query = evaluate_biencoder_model(model_name, batch_size)
    ablation_a_rows.append(row)
    ablation_a_per_query[model_name] = per_query

ablation_a_df = pd.DataFrame(ablation_a_rows)
display(ablation_a_df)


Loading bi-encoder: BAAI/bge-base-en-v1.5


Encoded 41,328 docs in 363.94s (113.6 docs/sec).
Embedding matrix memory: 0.118 GB


,model,indexing_time_sec,embedding_memory_gb,R@5_reranked,MRR_reranked,median_latency_ms
0,BAAI/bge-small-en-v1.5,134.368711,0.05912,0.9375,0.71875,60.400343
1,BAAI/bge-base-en-v1.5,363.936933,0.11824,0.8750,0.71875,70.688188


### 7B. Ablation B: Top-k Sweep for Re-ranker

The bi-encoder, cross-encoder, and generator choice are held constant. Only the number of bi-encoder candidates sent to the re-ranker changes.


In [ ]:
ablation_b_rows = []
for k_value in [5, 10, 20]:
    title_lists = {}

    def retrieve_then_rerank_for_latency(query):
        return rerank(query, retrieve(query, k=k_value), n=RERANK_N)

    for item in TEST_SET:
        ranked = retrieve_then_rerank_for_latency(item["query"])
        title_lists[item["query"]] = [record["title"] for record in ranked]

    summary, per_query = summarize_retrieval_metrics(title_lists, ks=(5,), mrr_k=5)
    latency = median_latency_ms(retrieve_then_rerank_for_latency, [item["query"] for item in TEST_SET])
    ablation_b_rows.append(
        {
            "bi_encoder_candidate_k": k_value,
            "R@5_reranked": summary["Recall@5"],
            "MRR_reranked": summary["MRR"],
            "median_retrieval_plus_rerank_latency_ms": latency,
        }
    )

ablation_b_df = pd.DataFrame(ablation_b_rows)
display(ablation_b_df)


,bi_encoder_candidate_k,R@5_reranked,MRR_reranked,median_retrieval_plus_rerank_latency_ms
0,5,0.8750,0.770833,30.057699
1,10,0.9375,0.723958,36.069458
2,20,0.9375,0.718750,61.181735


### 7C. Ablation C: Prompt Engineering

This compares the permissive prompt and strict prompt while holding retrieval, re-ranking, and the generator model constant. The year guard is disabled in this ablation so the prompt itself is the changed variable.


In [ ]:
def is_refusal(answer: str) -> bool:
    lower = str(answer).lower()

    refusal_markers = [
        "don't have information",
        "do not have information",
        "i don't know",
        "insufficient",


        "not enough information",
        "cannot answer",


    ]
    return any(marker in lower for marker in refusal_markers)


def run_prompt_on_test_set(prompt_name, system_prompt):
    rows = []

    for item in tqdm(TEST_SET, desc=f"Generating with {prompt_name} prompt"):
        result = run_rag_pipeline(

            item["query"],
            k=RETRIEVE_K,
            n=RERANK_N,
            system_prompt=system_prompt,
            use_year_guard=False,

        )
        rows.append(
            {

                "prompt": prompt_name,
                "query": item["query"],
                "category": item["category"],
                "gold_titles": item["gold_titles"],
                "reference_answer": item["reference_answer"],
                "answer": result["answer"],
                "retrieved_titles": result["retrieved_titles"],
                "contexts": result["contexts_text"],
                "is_unanswerable": len(item["gold_titles"]) == 0,
                "is_refusal": is_refusal(result["answer"]),

            }
        )
    return pd.DataFrame(rows)


prompt_outputs = {
    "permissive": run_prompt_on_test_set("permissive", PERMISSIVE_SYSTEM_PROMPT),
    "strict": run_prompt_on_test_set("strict", STRICT_SYSTEM_PROMPT),
}

ablation_c_hallucination_rows = []

for prompt_name, df_out in prompt_outputs.items():

    unanswerable = df_out[df_out["is_unanswerable"]]
    
    hallucinated_count = int((~unanswerable["is_refusal"]).sum())
    ablation_c_hallucination_rows.append(
        {
            "prompt": prompt_name,
            "unanswerable_queries": len(unanswerable),
            "hallucinated_answers_on_unanswerable": hallucinated_count,
        }
    )

ablation_c_hallucinations_df = pd.DataFrame(ablation_c_hallucination_rows)
display(ablation_c_hallucinations_df)

diff_rows = []
merged_prompt_outputs = prompt_outputs["permissive"].merge(
    prompt_outputs["strict"],
    on="query",
    suffixes=("_permissive", "_strict"),
)
for _, row in merged_prompt_outputs.iterrows():
    if row["answer_permissive"] != row["answer_strict"]:
        diff_rows.append(
            {
                "query": row["query"],
                "permissive_output": row["answer_permissive"],
                "strict_output": row["answer_strict"],
            }
        )
    if len(diff_rows) >= 2:
        break

ablation_c_examples_df = pd.DataFrame(diff_rows)
display(ablation_c_examples_df)


,prompt,unanswerable_queries,hallucinated_answers_on_unanswerable
0,permissive,4,4
1,strict,4,0


,query,permissive_output,strict_output
0,What is the plot of The Matrix?,The plot of The Matrix involves a computer hac...,The plot of The Matrix involves a computer hac...
1,Who directed Pulp Fiction?,The director of Pulp Fiction was Quentin Taran...,Quentin Tarantino [Pulp Fiction]


In [ ]:
# RAGAS helper. Official evaluate() docs for the pinned API:
# https://docs.ragas.io/en/v0.2.9/references/evaluate/

# Fast local RAGAS mode:
# Full 20-query x multi-metric RAGAS with a local judge can take hours on Colab T4.
# We keep the full 20-query retrieval metrics and hallucination counts, but run
# local RAGAS on a balanced diagnostic subset and document that choice.
RAGAS_EVAL_QUERIES = [
    "Who directed Inception?",
    "What is the plot of The Matrix?",
    "Which film is about dreams, corporate secrets, and a thief who enters people's minds?",
    "Who directed Toy Story?",
    "Who directed Oppenheimer (2023)?",
    "What is the plot of Dune: Part Two?",
]
RAGAS_CONTEXTS_PER_QUERY = 3
RAGAS_CONTEXT_CHAR_LIMIT = 650

ragas_judge_tokenizer = None
ragas_judge_model = None
ragas_judge_llm = None
ragas_embeddings = None


def ensure_ragas_judge():
    global ragas_judge_tokenizer, ragas_judge_model, ragas_judge_llm, ragas_embeddings
    if ragas_judge_llm is not None and ragas_embeddings is not None:
        return ragas_judge_llm, ragas_embeddings

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
    from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

    unload_generator()
    free_memory()

    judge_model_name = RAGAS_JUDGE_MODEL_NAME if torch.cuda.is_available() else CPU_FALLBACK_GENERATOR_MODEL_NAME
    print(f"Loading local RAGAS judge: {judge_model_name}")

    ragas_judge_tokenizer = AutoTokenizer.from_pretrained(judge_model_name, trust_remote_code=True)
    if ragas_judge_tokenizer.pad_token is None:
        ragas_judge_tokenizer.pad_token = ragas_judge_tokenizer.eos_token

    if torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        ragas_judge_model = AutoModelForCausalLM.from_pretrained(
            judge_model_name,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        ragas_judge_model = AutoModelForCausalLM.from_pretrained(
            judge_model_name,
            torch_dtype=torch.float32,
            trust_remote_code=True,
        )
        ragas_judge_model.to("cpu")

    judge_pipe = pipeline(
        "text-generation",
        model=ragas_judge_model,
        tokenizer=ragas_judge_tokenizer,
        max_new_tokens=RAGAS_JUDGE_MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
    )

    ragas_judge_llm = HuggingFacePipeline(pipeline=judge_pipe)
    ragas_embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_NAME,
        model_kwargs={"device": DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )
    return ragas_judge_llm, ragas_embeddings


def unload_ragas_judge():
    global ragas_judge_tokenizer, ragas_judge_model, ragas_judge_llm, ragas_embeddings
    ragas_judge_tokenizer = None
    ragas_judge_model = None
    ragas_judge_llm = None
    ragas_embeddings = None
    free_memory()


def compact_contexts_for_ragas(contexts):
    return [
        str(context)[:RAGAS_CONTEXT_CHAR_LIMIT]
        for context in list(contexts)[:RAGAS_CONTEXTS_PER_QUERY]
    ]


def run_ragas_for_outputs(outputs_df: pd.DataFrame, label: str, metric_names=None):
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import answer_relevancy, context_precision, faithfulness
    from ragas.run_config import RunConfig

    judge_llm, judge_embeddings = ensure_ragas_judge()

    metric_lookup = {
        "faithfulness": faithfulness,
        "context_precision": context_precision,
        "answer_relevancy": answer_relevancy,
    }
    metric_names = metric_names or ["faithfulness", "context_precision", "answer_relevancy"]
    selected_metrics = [metric_lookup[name] for name in metric_names]

    eval_df = outputs_df[outputs_df["query"].isin(RAGAS_EVAL_QUERIES)].copy()
    eval_df["contexts"] = eval_df["contexts"].apply(compact_contexts_for_ragas)

    print(f"RAGAS fast eval for {label}: {len(eval_df)} queries, metrics={metric_names}")

    dataset = Dataset.from_dict(
        {
            "question": eval_df["query"].tolist(),
            "answer": eval_df["answer"].tolist(),
            "contexts": eval_df["contexts"].tolist(),
            "ground_truth": eval_df["reference_answer"].tolist(),
        }
    )

    result = evaluate(
        dataset,
        metrics=selected_metrics,
        llm=judge_llm,
        embeddings=judge_embeddings,
        run_config=RunConfig(timeout=600, max_workers=1, max_retries=1, seed=SEED),
        raise_exceptions=False,
        show_progress=True,
        batch_size=1,
    )

    result_df = result.to_pandas()
    result_df.to_csv(f"ragas_{label}_row_scores.csv", index=False)
    mean_scores = result_df.mean(numeric_only=True).to_frame("mean_score").reset_index(names="metric")
    mean_scores["prompt"] = label
    return mean_scores, result_df


ragas_prompt_mean_rows = []
ragas_prompt_row_scores = {}

for prompt_name, df_out in prompt_outputs.items():
    try:
        means, rows = run_ragas_for_outputs(
            df_out,
            label=prompt_name,
            metric_names=["faithfulness"],
        )
        ragas_prompt_mean_rows.append(means)
        ragas_prompt_row_scores[prompt_name] = rows
    except Exception as exc:
        print(f"RAGAS failed for {prompt_name}: {type(exc).__name__}: {exc}")
        ragas_prompt_mean_rows.append(
            pd.DataFrame(
                {
                    "metric": ["faithfulness"],
                    "mean_score": [np.nan],
                    "prompt": [prompt_name],
                }
            )
        )

ragas_prompt_means_df = pd.concat(ragas_prompt_mean_rows, ignore_index=True)
display(ragas_prompt_means_df)

display(pd.DataFrame({"ragas_eval_subset_queries": RAGAS_EVAL_QUERIES}))

ablation_c_df = ablation_c_hallucinations_df.merge(
    ragas_prompt_means_df[ragas_prompt_means_df["metric"].eq("faithfulness")][["prompt", "mean_score"]],
    on="prompt",
    how="left",
).rename(columns={"mean_score": "ragas_faithfulness_mean"})

display(ablation_c_df)

# Keep the local judge loaded for Section 8 so the strict-prompt
# RAGAS metrics can reuse it without another model load.


Loading local RAGAS judge: Qwen/Qwen2.5-3B-Instruct


RAGAS fast eval for permissive: 6 queries, metrics=['faithfulness']


ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.executor:Exception raised in Job[1]: RagasOutputParserException(The output parser failed to parse the output including retries.)
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to pa

RAGAS fast eval for strict: 6 queries, metrics=['faithfulness']


ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.executor:Exception raised in Job[1]: RagasOutputParserException(The output parser failed to parse the output including retries.)
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to pa

,metric,mean_score,prompt
0,faithfulness,1.00,permissive
1,faithfulness,0.75,strict


,ragas_eval_subset_queries
0,Who directed Inception?
1,What is the plot of The Matrix?
2,"Which film is about dreams, corporate secrets,..."
3,Who directed Toy Story?
4,Who directed Oppenheimer (2023)?
5,What is the plot of Dune: Part Two?


,prompt,unanswerable_queries,hallucinated_answers_on_unanswerable,ragas_faithfulness_mean
0,permissive,4,4,1.00
1,strict,4,0,0.75


## 8. Test Set + Retrieval Metrics + RAGAS

This section reports the full custom retrieval metrics and the RAGAS means for the strict prompt.


In [ ]:
display(test_set_df[["category", "query", "gold_titles", "reference_answer"]])

bi_title_lists = {item["query"]: titles_from_bi(item["query"], k=10) for item in TEST_SET}
rerank_title_lists = {item["query"]: titles_from_reranker(item["query"], k=RETRIEVE_K, n=RERANK_N) for item in TEST_SET}

bi_summary, bi_per_query = summarize_retrieval_metrics(bi_title_lists, ks=(5, 10), mrr_k=10)
rerank_summary, rerank_per_query = summarize_retrieval_metrics(rerank_title_lists, ks=(5,), mrr_k=5)

custom_retrieval_metrics_df = pd.DataFrame(
    [
        {"configuration": "bi_encoder_only", **bi_summary},
        {"configuration": "bi_encoder_plus_cross_encoder", **rerank_summary},
    ]
)

display(custom_retrieval_metrics_df)

retrieved_titles_audit_df = pd.DataFrame(
    [
        {
            "query": item["query"],
            "category": item["category"],
            "gold_titles": item["gold_titles"],
            "bi_top10": bi_title_lists[item["query"]],
            "reranked_top5": rerank_title_lists[item["query"]],
        }
        for item in TEST_SET
    ]
)
display(retrieved_titles_audit_df)

strict_outputs_df = prompt_outputs["strict"].copy()
strict_outputs_df.to_csv("strict_prompt_end_to_end_outputs.csv", index=False)
display(strict_outputs_df[["query", "answer", "retrieved_titles"]])

strict_faithfulness_means_df = ragas_prompt_means_df[
    ragas_prompt_means_df["prompt"].eq("strict")
    & ragas_prompt_means_df["metric"].eq("faithfulness")
].copy()

try:
    strict_extra_ragas_means_df, strict_extra_ragas_rows_df = run_ragas_for_outputs(
        strict_outputs_df,
        label="strict",
        metric_names=["context_precision", "answer_relevancy"],
    )

    strict_ragas_means_df = pd.concat(
        [strict_faithfulness_means_df, strict_extra_ragas_means_df],
        ignore_index=True,
    )

except Exception as exc:
    print(f"Extra strict RAGAS metrics failed: {type(exc).__name__}: {exc}")
    strict_ragas_means_df = pd.concat(
        [
            strict_faithfulness_means_df,
            pd.DataFrame(
                {
                    "metric": ["context_precision", "answer_relevancy"],
                    "mean_score": [np.nan, np.nan],
                    "prompt": ["strict", "strict"],
                }
            ),
        ],
        ignore_index=True,
    )

display(strict_ragas_means_df)

unload_ragas_judge()


,category,query,gold_titles,reference_answer
0,standard,Who directed Inception?,[Inception],Christopher Nolan directed Inception.
1,standard,What is the plot of The Matrix?,[The Matrix],The Matrix is about a computer hacker who lear...
2,standard,Who directed Pulp Fiction?,[Pulp Fiction],Quentin Tarantino directed Pulp Fiction.
3,standard,Which movie features a clownfish named Nemo?,[Finding Nemo],Finding Nemo features the clownfish Nemo.
4,standard,Which movie follows a young lion prince reclai...,[The Lion King],The Lion King follows a young lion prince recl...
5,adversarial,"Which film is about dreams, corporate secrets,...",[Inception],"Inception is about dreams, corporate secrets, ..."
6,adversarial,What movie is about romance and disaster aboar...,[Titanic],Titanic is the romance and disaster movie aboa...
7,adversarial,Which film has a ring that must be destroyed i...,[The Lord of the Rings: The Fellowship of the ...,The Lord of the Rings: The Fellowship of the R...
8,adversarial,What movie is about a shark terrorizing Amity ...,[Jaws],Jaws is about a shark terrorizing Amity Island.
9,adversarial,Which film features a chocolate factory run by...,[Willy Wonka & the Chocolate Factory],Willy Wonka & the Chocolate Factory features W...


,configuration,Recall@5,Recall@10,MRR
0,bi_encoder_only,0.8750,0.9375,0.763021
1,bi_encoder_plus_cross_encoder,0.9375,NaN,0.718750


,query,category,gold_titles,bi_top10,reranked_top5
0,Who directed Inception?,standard,[Inception],"[Inception, Dream Work, Room 237, The American...","[Inception, Transcendent Man, Directed by Sidn..."
1,What is the plot of The Matrix?,standard,[The Matrix],"[The Matrix, The Matrix Reloaded, The Matrix R...","[The Matrix, The Matrix Revolutions, The Matri..."
2,Who directed Pulp Fiction?,standard,[Pulp Fiction],"[Pulp Fiction, Pulp: a Film About Life, Death ...","[Pulp Fiction, Pulp, Pulp: a Film About Life, ..."
3,Which movie features a clownfish named Nemo?,standard,[Finding Nemo],"[Finding Nemo, Devilfish, Little Nemo: Adventu...","[Finding Nemo, Little Nemo: Adventures In Slum..."
4,Which movie follows a young lion prince reclai...,standard,[The Lion King],"[The Brothers Lionheart, The Lion King, Vaesen...","[Vaesen, The Lion in Winter, The Little Norse ..."
5,"Which film is about dreams, corporate secrets,...",adversarial,[Inception],"[Inception, Thieves, Stolen Identity, Identity...","[Dreamscape, Inception, Identity Thief, Dream ..."
6,What movie is about romance and disaster aboar...,adversarial,[Titanic],"[Titanic, Titanica, Stormy Waters, The Chamber...","[A Night to Remember, Titanic, Titanica, Titan..."
7,Which film has a ring that must be destroyed i...,adversarial,[The Lord of the Rings: The Fellowship of the ...,"[The Return of the King, Stonehenge Apocalypse...","[The Return of the King, The Lord of the Rings..."
8,What movie is about a shark terrorizing Amity ...,adversarial,[Jaws],"[Jaws, Shark Attack 3: Megalodon, Shark Attack...","[Jaws, Jaws 2, Jaws: The Revenge, Sharktopus, ..."
9,Which film features a chocolate factory run by...,adversarial,[Willy Wonka & the Chocolate Factory],"[Willy Wonka & the Chocolate Factory, Charlie ...","[Willy Wonka & the Chocolate Factory, Epic Mov..."


,query,answer,retrieved_titles
0,Who directed Inception?,The director of Inception was Christopher Nola...,"[Inception, Transcendent Man, Directed by Sidn..."
1,What is the plot of The Matrix?,The plot of The Matrix involves a computer hac...,"[The Matrix, The Matrix Revolutions, The Matri..."
2,Who directed Pulp Fiction?,Quentin Tarantino [Pulp Fiction],"[Pulp Fiction, Pulp, Pulp: a Film About Life, ..."
3,Which movie features a clownfish named Nemo?,Finding Nemo [2003] [Finding Nemo],"[Finding Nemo, Little Nemo: Adventures In Slum..."
4,Which movie follows a young lion prince reclai...,[The Little Norse Prince],"[Vaesen, The Lion in Winter, The Little Norse ..."
5,"Which film is about dreams, corporate secrets,...",The film that matches the description of being...,"[Dreamscape, Inception, Identity Thief, Dream ..."
6,What movie is about romance and disaster aboar...,The movie that is about romance and disaster a...,"[A Night to Remember, Titanic, Titanica, Titan..."
7,Which film has a ring that must be destroyed i...,The Return of the King [1980] and The Lord of ...,"[The Return of the King, The Lord of the Rings..."
8,What movie is about a shark terrorizing Amity ...,The movie is [Jaws] or [Jaws 2]. Both movies f...,"[Jaws, Jaws 2, Jaws: The Revenge, Sharktopus, ..."
9,Which film features a chocolate factory run by...,Willy Wonka & the Chocolate Factory [1971] [Wi...,"[Willy Wonka & the Chocolate Factory, Epic Mov..."


RAGAS fast eval for strict: 6 queries, metrics=['context_precision', 'answer_relevancy']


ERROR:ragas.executor:Exception raised in Job[0]: ValidationError(2 validation errors for Verification
reason
  Field required [type=missing, input_value={'text': '{\n    "reason"...,\n    "verdict": 0\n}'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
verdict
  Field required [type=missing, input_value={'text': '{\n    "reason"...,\n    "verdict": 0\n}'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing)
ERROR:ragas.executor:Exception raised in Job[6]: ValidationError(2 validation errors for Verification
reason
  Field required [type=missing, input_value={'text': '{"reason": "The... film.", "verdict": 0}'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
verdict
  Field required [type=missing, input_value={'text': '{"reason": "The... film.", "verdict": 0}'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missi

,metric,mean_score,prompt
0,faithfulness,0.750000,strict
1,context_precision,0.625000,strict
2,answer_relevancy,0.663007,strict


## 9. Failure Diary / Bad Cases Table

The first two rows include implemented before/after fixes. The last two document residual failures or limitations that remain visible after the final pipeline.


In [ ]:
failure_rows = []

# Fix 1: generation context without director/cast metadata cannot answer director questions reliably.
q1 = "Who directed Inception?"
q1_contexts = search(q1)
q1_before = generate_answer(
    q1,
    q1_contexts,
    system_prompt=STRICT_SYSTEM_PROMPT,
    context_text=format_context_overview_only(q1_contexts),
)
q1_after = generate_answer(q1, q1_contexts, system_prompt=STRICT_SYSTEM_PROMPT)
failure_rows.append(
    {
        "Input / Query": q1,
        "Model Output": q1_before,
        "Gold / Correct Answer": "Christopher Nolan directed Inception.",
        "Error Type": "Metadata Absence",
        "Root Cause Analysis": "The retrieved movie was correct, but an overview-only context did not expose the director field needed by the question.",
        "Specific Fix Implemented": "Included Director, Year, Genres, and Cast fields in the generator context.",
        "After Fix Output": q1_after,
        "Evidence": "The same retrieved title can now support the director answer from context.",
    }
)

# Fix 2: permissive prompt on post-cutoff questions often answers from model memory.
q2 = "Who directed Oppenheimer (2023)?"
q2_before = run_rag_pipeline(q2, system_prompt=PERMISSIVE_SYSTEM_PROMPT, use_year_guard=False)
q2_after = run_rag_pipeline(q2, system_prompt=STRICT_SYSTEM_PROMPT, use_year_guard=True)
failure_rows.append(
    {
        "Input / Query": q2,
        "Model Output": q2_before["answer"],
        "Gold / Correct Answer": "Refuse: Oppenheimer (2023) is outside the pre-July 2017 corpus.",
        "Error Type": "Hallucination / Prompt Leakage",
        "Root Cause Analysis": "A permissive prompt lets the local LLM use parametric knowledge when retrieved context is not sufficient.",
        "Specific Fix Implemented": "Switched to the strict refusal prompt and added a deterministic post-2017 year guard in run_query.",
        "After Fix Output": q2_after["answer"],
        "Evidence": "Hallucinated answer count on unanswerable queries is reported in Ablation C; this query becomes a refusal.",
    }
)

# Residual failure 1: duplicate-title cleaning can hide a later movie with the same exact title.
q3 = "Who directed The Avengers (2012)?"
q3_out = run_rag_pipeline(q3)
failure_rows.append(
    {
        "Input / Query": q3,
        "Model Output": q3_out["answer"],
        "Gold / Correct Answer": "Joss Whedon directed the 2012 movie The Avengers.",
        "Error Type": "Retrieval Miss / Duplicate Title Collision",
        "Root Cause Analysis": "The duplicate-title cleaning step keeps one row per title by longer overview; for same-title remakes/franchise entries, the requested year may have been removed.",
        "Specific Fix Implemented": "Not applied in final pipeline; preprocessing deliberately deduplicates by title only.",
        "After Fix Output": "",
        "Evidence": f"Retrieved titles: {q3_out['retrieved_titles']}",
    }
)

# Residual failure 2: some plot-detail questions need details absent from the TMDB overview.
q4 = "What is the name of the hotel in The Shining?"
q4_out = run_rag_pipeline(q4)
failure_rows.append(
    {
        "Input / Query": q4,
        "Model Output": q4_out["answer"],
        "Gold / Correct Answer": "The hotel is the Overlook Hotel.",
        "Error Type": "Metadata Absence / Corpus Coverage",
        "Root Cause Analysis": "The corpus stores one short overview per movie; fine-grained details can be absent even when the correct movie is retrieved.",
        "Specific Fix Implemented": "Not applied; a production fix would add richer plot summaries or chunk external synopses, which is outside this dataset-only project.",
        "After Fix Output": "",
        "Evidence": f"Retrieved titles: {q4_out['retrieved_titles']}",
    }
)

failure_diary_df = pd.DataFrame(failure_rows)
display(failure_diary_df)
failure_diary_df.to_csv("failure_diary.csv", index=False)

Loading local generator: Qwen/Qwen2.5-3B-Instruct


,Input / Query,Model Output,Gold / Correct Answer,Error Type,Root Cause Analysis,Specific Fix Implemented,After Fix Output,Evidence
0,Who directed Inception?,I don't have information about that in the pro...,Christopher Nolan directed Inception.,Metadata Absence,"The retrieved movie was correct, but an overvi...","Included Director, Year, Genres, and Cast fiel...",The director of Inception was Christopher Nola...,The same retrieved title can now support the d...
1,Who directed Oppenheimer (2023)?,Based on the information provided in Document ...,Refuse: Oppenheimer (2023) is outside the pre-...,Hallucination / Prompt Leakage,A permissive prompt lets the local LLM use par...,Switched to the strict refusal prompt and adde...,I don't have information about that in the pro...,Hallucinated answer count on unanswerable quer...
2,Who directed The Avengers (2012)?,I don't have information about that in the pro...,Joss Whedon directed the 2012 movie The Avengers.,Retrieval Miss / Duplicate Title Collision,The duplicate-title cleaning step keeps one ro...,Not applied in final pipeline; preprocessing d...,,"Retrieved titles: ['The Avengers', 'Avengers: ..."
3,What is the name of the hotel in The Shining?,The hotel in The Shining is called the Overloo...,The hotel is the Overlook Hotel.,Metadata Absence / Corpus Coverage,The corpus stores one short overview per movie...,Not applied; a production fix would add richer...,,"Retrieved titles: ['The Shining', 'Room 237', ..."


## 10. Discussion & Results

### 10.1 System summary

The final system is a fully local RAG pipeline over The Movies Dataset with a pre-July 2017 cutoff. Documents are embedded with `BAAI/bge-small-en-v1.5` and stored in a FAISS `IndexFlatIP` over L2-normalized vectors. The cross-encoder `cross-encoder/ms-marco-MiniLM-L-6-v2` re-ranks the bi-encoder's top 20 down to 5 contexts, and `Qwen/Qwen2.5-3B-Instruct` in 4-bit NF4 generates the answer. RAGAS uses a local judge; the run reported here used `Qwen/Qwen2.5-3B-Instruct` in 4-bit (see the Section 7C output). No hosted API is used for retrieval, re-ranking, generation or evaluation.

### 10.2 Data pipeline

The full preprocessed corpus is indexed with no random subsampling, because subsampling can drop franchise entries and break sequel queries. Preprocessing drops rows with a missing title or an empty overview, applies the 31 July 2017 cutoff, drops duplicate titles (keeping the row with the longer overview), and merges `credits.csv` on `movie_id` to add the director (crew rows with `job == "Director"`) and the top-5 cast. Each movie becomes one `document_text`.

The raw `movies_metadata.csv` has 45,466 rows and the final corpus has 41,328 movies. Overviews are short: mean 56.40 whitespace tokens, median 50, max 187. This matters later, because some plot-detail questions cannot be answered from a one-paragraph overview.

`document_text` puts each field on its own line in the order Title, Year, Genres, Director, Cast, Overview. Title and year help with sequel confusion, and the director and cast fields let "who directed X" questions be answered from retrieved context instead of the LLM's parametric memory. The three printed examples (*!Women Art Revolution*, 2010; *Mothra vs. Godzilla*, 1964, directed by Ishirô Honda; *ファンタスティポ*, 2005) all follow this structure, and the Japanese title confirms that non-ASCII text passes through cleanly.

### 10.3 Embedding and indexing

Documents and queries are L2-normalized at encoding time, so inner product in `IndexFlatIP` equals cosine similarity and there is no mismatch between index-time and query-time normalization.

FAISS was chosen over Chroma because at 41k vectors and 384 dimensions exact search is already fast enough that no ANN index is needed, and FAISS makes indexing time, query latency and embedding memory easy to measure for the ablations. The trade-off is that FAISS stores no metadata, so a `records.jsonl` file sits next to each index with one row per vector, kept in sync by insertion order. A production system that needs metadata filtering would be more comfortable on Chroma.

Encoding all 41,328 documents took 134.37 s (307.6 docs/sec on a T4). The 384-dimensional embedding matrix uses 0.059 GB. Indexes are persisted under `rag_indexes/`, so reruns load them from disk instead of re-encoding.

Sanity check, "Who directed Inception?": *Inception* (2010) is rank 1 at 0.7558, ahead of *Dream Work* (0.6627), *Room 237* (0.6515), *The American Dreamer* (0.6434) and *Dreams on Spec* (0.6406). The correct movie leads by a clear margin. The four distractors are all dream-themed, which is expected from a sentence embedding that weights topical words more than proper nouns.

### 10.4 Retrieval and re-ranking

`retrieve(query, k=20)` returns the bi-encoder's top 20, and `rerank(query, candidates, n=5)` rescores them with the cross-encoder and keeps the top 5. The three comparison queries test sequel confusion, adversarial wording and franchise disambiguation.

- **"Who directed Toy Story?"** The bi-encoder ranks *Toy Story 3* (0.7242) and *Toy Story 2* (0.7075) above the 1995 original (0.6955), most likely because the sequels' longer overviews push their similarity up. After re-ranking, *Toy Story* (1995) moves to rank 1 (7.9102), followed by *Toy Story 2* (7.8594), *Toy Story 3* (7.4759), *Toy Story That Time Forgot* (6.9290) and *Toy Story of Terror!* (6.7335). The bi-encoder scores are clustered around 0.7; the cross-encoder separates them much more clearly.
- **"Which film is about a thief who enters dreams to steal secrets?"** Neither stage finds *Inception*: the "thief / steal" vocabulary dominates "dreams". The bi-encoder returns *Thieves*, *Yolanda and the Thief*, *Stolen Identity*, *Thief* and *Thieves After Dark*. After re-ranking, only *To Steal from a Thief* scores positive (1.0784); *Seven Thieves* (−1.6570), *Thieves* (−1.8678), *Once a Thief* (−2.0500) and *Stolen* (−2.0937) are negative, which signals that no candidate is a strong match.
- **"What movie is about a shark terrorizing Amity Island?"** The bi-encoder already ranks *Jaws* (1975) first (0.7915), followed by generic shark films (*Shark Attack 3: Megalodon*, *Shark Attack*, *Jaws 2*, *Shark Attack 2*). Re-ranking keeps *Jaws* first (6.6928) and promotes *Jaws 2* (4.2847) and *Jaws: The Revenge* (4.1490) above the generic titles, because "Amity Island" is specific to the Jaws franchise. Even when the top answer does not change, re-ranking makes the context passed to the generator more relevant.

### 10.5 Generation

The generator is `Qwen/Qwen2.5-3B-Instruct` with 4-bit NF4 quantization (bitsandbytes). It fits on a Colab T4 alongside the bi-encoder and cross-encoder and generally follows the `[Title]` citation format and the exact refusal string. Decoding is greedy (`do_sample=False`, `max_new_tokens=180`), so the before/after comparisons in the failure diary are deterministic. A CPU fallback to `Qwen2.5-0.5B-Instruct` exists only so the notebook can be smoke-tested without a GPU.

The strict system prompt requires the model to:

1. answer only from the provided movie context;
2. reply with exactly "I don't have information about that in the provided context." when the context is insufficient;
3. never use outside knowledge, even for well-known movies;
4. cite every movie title it uses in square brackets, e.g. `[Toy Story]`;
5. refuse questions about movies released after July 2017 unless that exact movie appears in the context.

It ends with a one-line refusal example (*Oppenheimer*, 2023), added because without it the model sometimes still attempted post-2017 questions.

A separate year guard in `run_rag_pipeline()` returns the refusal string before generation whenever the query mentions a year after 2017. In Ablation C, with the guard disabled, the strict prompt alone already reaches 0/4 hallucinations on unanswerable queries; the guard is a deterministic backstop.

On the five generation examples, the four answerable queries return grounded answers with bracket citations (for example "The director of Inception was Christopher Nolan. [Inception]" and "The director of Toy Story is John Lasseter. [Toy Story]"), and "Who directed Oppenheimer (2023)?" returns the exact refusal string.

### 10.6 `run_query` self-test

`run_query()` wraps the pipeline in `try/except` and always returns a dict with `answer` and `retrieved_titles`; on an internal error it returns the refusal string and an empty title list. The self-test passes on all four inputs:

- "Who directed Inception?": 5 titles and the Nolan answer
- "asdfgh nonsense query": 5 titles and the refusal string
- the empty string: 0 titles and the refusal string, because empty or whitespace-only input short-circuits before the bi-encoder
- the Turkish query "Bir Zamanlar Anadolu'da hakkinda ne biliyorsun?": 5 titles and the refusal string

### 10.7 Ablations

**A. Bi-encoder size.** Cross-encoder, `k`, `n`, generator and test set are held constant.

| Model | Indexing time (s) | Embedding memory (GB) | R@5 (re-ranked) | MRR (re-ranked) | Median latency (ms) |
|---|---|---|---|---|---|
| BAAI/bge-small-en-v1.5 | 134.37 | 0.059 | 0.9375 | 0.7188 | 60.40 |
| BAAI/bge-base-en-v1.5 | 363.94 | 0.118 | 0.8750 | 0.7188 | 70.69 |

The base model was expected to do better, but on this test set the small model scores higher R@5 and ties on MRR, with less than half the indexing time and half the memory. Overviews are short (median 50 tokens) and the discriminative tokens are mostly proper nouns (titles, director names), which both models embed well, so the base model's extra capacity does not pay off on this corpus.

**B. Re-ranker candidate pool (`k`).** Only the bi-encoder pool size changes; the re-ranker still keeps the top 5.

| Bi-encoder k | R@5 (re-ranked) | MRR (re-ranked) | Median retrieval + re-rank latency (ms) |
|---|---|---|---|
| 5 | 0.8750 | 0.7708 | 30.06 |
| 10 | 0.9375 | 0.7240 | 36.07 |
| 20 | 0.9375 | 0.7188 | 61.18 |

Going from k=5 to k=10 lets the re-ranker see one more query's gold title, so R@5 rises. Going from k=10 to k=20 adds no recall but raises median latency from 36.07 ms to 61.18 ms. MRR falls slightly as k grows: a larger pool gives the cross-encoder more chances to push a strong distractor above the gold title. For this system k=10 is the best trade-off.

**C. Prompt engineering.** Retrieval, re-ranking and the generator are held constant, and the year guard is disabled so the system prompt is the only variable.

| Prompt | Unanswerable queries | Hallucinated answers | RAGAS faithfulness (6-query subset) |
|---|---|---|---|
| Permissive | 4 | 4 / 4 | 1.00 |
| Strict | 4 | 0 / 4 | 0.75 |

The permissive prompt scores higher faithfulness yet fails to refuse on every unanswerable query. RAGAS faithfulness only checks whether the answer's claims are supported by the retrieved context, so a confident answer about the wrong movie, grounded in a distractor document, can still score well. For example, on "Who directed Oppenheimer (2023)?" the permissive prompt answered with the director of *The Day After Trinity*, a documentary retrieved as a near match, instead of refusing. The hallucination count is the metric that matters here, and the strict prompt brings it from 4 to 0.

In the side-by-side examples, "Who directed Pulp Fiction?" gets "The director of Pulp Fiction was Quentin Tarantino. [Pulp Fiction]" from the permissive prompt and the shorter "Quentin Tarantino [Pulp Fiction]" from the strict prompt. On "What is the plot of The Matrix?" both prompts summarize the plot, but the permissive answer continues with a longer unprompted elaboration.

### 10.8 Retrieval metrics

Recall@5, Recall@10 and MRR are computed over the 16 answerable queries; the 4 unanswerable queries have no gold titles and are excluded.

| Configuration | Recall@5 | Recall@10 | MRR |
|---|---|---|---|
| Bi-encoder only | 0.8750 | 0.9375 | 0.7630 (top 10) |
| Bi-encoder + cross-encoder | 0.9375 | n/a | 0.7188 (top 5) |

Re-ranking raises Recall@5 by one query, which matters because the generator only sees the top 5 contexts. MRR is lower after re-ranking (note the different cutoffs), the same effect seen in Ablation B: the cross-encoder reorders more aggressively, and on some queries where the bi-encoder already had the gold title at rank 1 it places another candidate above it. In the audit table, *The Godfather: Part III* moves above *The Godfather* and *A Night to Remember* above *Titanic*. The gold title stays in the top 5 (Recall@5 improves) but is not always at rank 1 (MRR drops).

### 10.9 RAGAS evaluation

RAGAS was run on a balanced 6-query subset (2 standard, 1 adversarial, 1 sequel-confusion, 2 unanswerable) rather than all 20, because full multi-metric evaluation with a local judge was too slow on a T4. The retrieval metrics and hallucination counts above still use the full 20-query set.

On the strict prompt, RAGAS reports faithfulness 0.750, context precision 0.625 and answer relevancy 0.663. Context precision is the weakest metric. That matches the retrieved-titles audit: on adversarial queries the re-ranked context contains topically related distractors (for the dreams / corporate-secrets query, *Dreamscape* is ranked above *Inception*). Answer relevancy is likely pulled down by the refusals on unanswerable queries, which a judge can score as not addressing the question even though refusing is the correct behavior.

**Reliability caveats.** `Qwen/Qwen2.5-7B-Instruct` was tried first as the judge but ran out of memory on the T4, so the judge is `Qwen/Qwen2.5-3B-Instruct` in 4-bit (RAGAS 0.2.9; metrics: faithfulness, context_precision, answer_relevancy). The smaller judge often returned output that RAGAS could not parse. The stderr kept in Sections 7C and 8 shows the failure modes:

- `Prompt fix_output_format / n_l_i_statement_prompt / long_form_answer_prompt failed to parse output`
- `RagasOutputParserException` for several jobs
- `AttributeError('StringIO' object has no attribute 'statements')`
- `ValidationError` on missing `reason` / `verdict` fields on the `context_precision` path, with "All inputs must have verdict attribute" warnings

Rows that fail to parse are scored as NaN and excluded from the means, so the RAGAS numbers should be read as indicative rather than definitive. The hallucination count on unanswerable queries is the more reliable signal of grounding.

### 10.10 Failure diary

The first two cases have implemented before/after fixes; the last two are residual limitations.

| Query | Before | Error type | Root cause | Fix | After |
|---|---|---|---|---|---|
| Who directed Inception? | Refusal string | Metadata absence | The correct movie was at rank 1, but an overview-only context never showed the director to the LLM. | Added Director, Year, Genres and Cast to `document_text` and to the generator context (`format_context()`). | "The director of Inception was Christopher Nolan. [Inception]" |
| Who directed Oppenheimer (2023)? | Permissive prompt: answered with the director of *The Day After Trinity* (a retrieved near match) instead of refusing | Hallucination / prompt leakage | The permissive prompt does not require a refusal when the context lacks the asked-about movie. | Strict prompt (context-only, exact refusal string, refusal example) plus the post-2017 year guard. | Refusal string |
| Who directed The Avengers (2012)? | Refusal string | Retrieval miss / duplicate-title collision | Deduplication by title keeps one row per title (the longer overview), so the 1998 *The Avengers* survives and the 2012 film is dropped. | Not applied; the fix would be to deduplicate on (title, year). | n/a |
| What is the name of the hotel in The Shining? | "The hotel in The Shining is called the Overlook Hotel. [The Shining]" | Corpus coverage probe | Logged as a probe for fine-grained plot questions. This overview does mention the Overlook Hotel, so the answer is grounded, but details absent from the one-paragraph overviews cannot be answered from this corpus. | Not applied; would need richer plot summaries or external synopses. | n/a |

The Inception fix changed a refusal into a correct, cited answer with the same retrieval. The Oppenheimer fix turned a non-refusal into the exact refusal string, and the same prompt change drives the Ablation C result (hallucinations on unanswerable queries from 4/4 to 0/4).

### 10.11 Reproducibility

- Seed 455 everywhere (Python, NumPy, PyTorch, RAGAS `RunConfig.seed`)
- Retrieval pool k=20, re-ranked contexts n=5
- Embedding batch size 128 (`bge-small`) and 64 (`bge-base`); cross-encoder batch size 32
- Generator: greedy decoding, `max_new_tokens=180`
- RAGAS judge: `max_new_tokens=512`, `max_workers=1`, `timeout=600`
- FAISS indexes persist under `rag_indexes/`, so the encoding cost is paid once

### 10.12 Lessons learned

- **RAG failures are quiet.** The system rarely crashes; it retrieves a sequel and answers confidently about the wrong movie. *Toy Story* is the clearest example: the bi-encoder ranked *Toy Story 3* above the original, and only the cross-encoder fixed it.
- **Document format matters as much as model choice.** Adding director and cast turned the Inception query from a refusal into a correct answer with no change to retrieval.
- **Faithfulness is not correctness.** The permissive prompt scored 1.00 on RAGAS faithfulness while failing to refuse on 4/4 unanswerable queries, so faithfulness alone is not a sufficient check.
- **Bigger embeddings are not automatically better.** On this corpus `bge-small` beat `bge-base` on R@5 with half the memory.
- **Re-ranking needs the right pool size.** k=10 matched k=20 on R@5 at lower latency.
- **Refusals benefit from two layers.** The strict prompt handles unanswerable questions in Ablation C, and the code-side year guard adds a deterministic backstop.